# `genimg_x402_token` Buyer Notebook — real end-to-end image payment (Deno/TypeScript)

Drives the **real, locally-running** `genimg_x402_token.ts` handler as a buyer would, over plain
HTTP via `wrapFetchWithPayment` — exactly what a real client does. The server decides and
advertises everything else.

Two jobs:

1. **Verify the `images/v1` envelope over a real payment.** The unit tests cover its shape with
   mocks; nothing has exercised it against a real facilitator.
2. **Be the blueprint for the frontend work**, the way `sc_llm_x402_buyer.ipynb` was for
   `useX402Chat.ts`. `website/components/ImageGenerator.tsx` still reads the old flat shape.

Shorter than the chat buyer notebook, because `exact` is stateless: no channel storage, no
deposit strategy, no corrective-402 recovery. Just `registerExactEvmScheme`.

## Prerequisites

- `cd scw_js && npm run dev:x402` (port 8082)
- `TEST_WALLET_PRIVATE_KEY` in `scw_js/.env`, funded with **Optimism Sepolia USDC** ($0.07/call)
- **The server wallet (`NFT_WALLET_PRIVATE_KEY`) needs Optimism Sepolia ETH** — the mint is real
  even on testnet. `preFlightChecks` returns a 500 naming the deficit if it is short. This is the
  easiest prerequisite to overlook.
- No local facilitator needed: `FACILITATOR_URL` defaults to the deployed one.

In [ ]:
// Setup: imports + envimport { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";import { privateKeyToAccount } from "npm:viem@2/accounts";// scw_js's own .env, one level up — the single source of truth (buyer key, receiver address,// and the server's own secrets all live here; the server reads the same file).const env = await load({ envPath: "../.env", examplePath: null, export: true });const PRIVATE_KEY = env.TEST_WALLET_PRIVATE_KEY;if (!PRIVATE_KEY) {    throw new Error("TEST_WALLET_PRIVATE_KEY missing from scw_js/.env — add a key funded with Optimism Sepolia USDC.");}const account = privateKeyToAccount(`0x${PRIVATE_KEY.replace(/^0x/, "")}`);console.log("🎨 genimg_x402_token buyer notebook");console.log(`   Buyer (payer): ${account.address}`);console.log(`   NFT recipient: derived from the payment payload — no recipient field is sent`);

## Network selection

**Note the inversion from `sc_llm_x402_buyer.ipynb`.** That notebook's testnet is *Base* Sepolia,
because Optimism Sepolia has no batch-settlement contract. genimg is the mirror image: GenImNFT is
deployed on **Optimism Sepolia only** (`TESTNET_GENAI_NFT_ADDRESSES` in
`shared/chain-utils/src/addresses.ts`), so Base Sepolia is the unsupported one here. Copying that
notebook's flag cell picks the wrong chain.

| `USE_MAINNET` | `USE_BASE` | Network | CAIP-2 | Real image? |
|---|---|---|---|---|
| `false` | `false` | Optimism Sepolia | `eip155:11155420` | No — the server always mocks (`isTestnet`), returning a placeholder |
| `false` | `true` | — | — | **Unsupported**: no GenImNFT on Base Sepolia (the cell throws) |
| `true` | `false` | Optimism Mainnet | `eip155:10` | **Yes** — real USDC settlement + a real, billed BFL generation |
| `true` | `true` | Base Mainnet | `eip155:8453` | **Yes** — same |

On testnet the *image* is free, but the **mint and the USDC settlement are real**.

In [ ]:
import { base, optimism, optimismSepolia } from "npm:viem@2/chains";const USE_MAINNET = false;  // ⚠️ true = REAL MONEY: mainnet USDC settlement + a real, billed BFL generation.const USE_BASE = false;     // true = Base, false = Optimism. (Base is mainnet-only here, see guard.)// Guard: GenImNFT is not deployed on Base Sepolia, so the server could never mint there.// Fail loudly rather than at a confusing 402 three cells later.if (USE_BASE && !USE_MAINNET) {    throw new Error(        "Base Sepolia is unsupported for genimg (no GenImNFT deployment). " +        "Use Optimism Sepolia (USE_BASE = false) or a mainnet run (USE_MAINNET = true).",    );}const NETWORK_CONFIG = {    "optimism-testnet": {        caip2Network: "eip155:11155420" as const, networkName: "Optimism Sepolia (Testnet)",        chain: optimismSepolia,        // EIP-712 domain name differs by network — testnet USDC is "USDC", mainnet is        // "USD Coin". Getting this wrong silently breaks signature verification.        usdcName: "USDC",        usdcAddress: "0x5fd84259d66Cd46123540766Be93DFE6D43130D7" as `0x${string}`,        nftContract: "0x10827cC42a09D0BAD2d43134C69F0e776D853D85",        rpcUrl: undefined as string | undefined,        explorer: "https://sepolia-optimism.etherscan.io", faucet: "https://faucet.circle.com/",    },    "optimism-mainnet": {        caip2Network: "eip155:10" as const, networkName: "Optimism Mainnet",        chain: optimism, usdcName: "USD Coin",        usdcAddress: "0x0b2C639c533813f4Aa9D7837CAf62653d097Ff85" as `0x${string}`,        nftContract: "0x80f95d330417a4acEfEA415FE9eE28db7A0A1Cdb",        rpcUrl: "https://optimism-rpc.publicnode.com",        explorer: "https://optimistic.etherscan.io", faucet: "Bridge: https://app.optimism.io/bridge",    },    "base-mainnet": {        caip2Network: "eip155:8453" as const, networkName: "Base Mainnet",        chain: base, usdcName: "USD Coin",        usdcAddress: "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913" as `0x${string}`,        nftContract: "0xa5d6a3eEDADc3346E22dF9556dc5B99f2777ab68",        rpcUrl: "https://base-rpc.publicnode.com",        explorer: "https://basescan.org", faucet: "Bridge: https://bridge.base.org",    },};const configKey = USE_MAINNET ? (USE_BASE ? "base-mainnet" : "optimism-mainnet") : "optimism-testnet";const config = NETWORK_CONFIG[configKey];const NETWORK = config.caip2Network;// Orthogonal to USE_MAINNET — same code path either way.// Local: `cd scw_js && npm run dev:x402`.const USE_DEPLOYED = false;const SERVICE_URL = USE_DEPLOYED ? "https://imagegen-agent.fretchen.eu" : "http://localhost:8082";console.log(USE_MAINNET ? `🚨 REAL MONEY on ${config.networkName}` : `🧪 ${config.networkName}`);console.log(`   ${NETWORK} • USDC ${config.usdcName} @ ${config.usdcAddress}`);console.log(`   GenImNFT: ${config.nftContract}`);console.log(`   Service : ${SERVICE_URL}`);

## Buyer setup

This is the whole difference from the chat notebook. `exact` ships a client-register helper, so
there is no scheme to construct by hand, no `ClientChannelStorage`, no deposit strategy, and no
corrective-402 path — the payment is a one-shot EIP-3009 authorization, not a channel.

`@x402/*` is floored at **2.24.0**, the version `scw_js` currently resolves. The caret still lets
Deno float to a newer minor, which is normally fine — but x402 minors have changed verify
behaviour before, so if a run fails in a way the server's own tests do not reproduce, pin the
exact version here first.

In [ ]:
import { x402Client, wrapFetchWithPayment, x402HTTPClient } from "npm:@x402/fetch@^2.24.0";import { registerExactEvmScheme } from "npm:@x402/evm@^2.24.0/exact/client";import { createPublicClient, http } from "npm:viem@2";const publicClient = createPublicClient({ chain: config.chain, transport: http(config.rpcUrl) });// The signer shape `useX402ImageGeneration.ts` builds from a wagmi WalletClient: an address plus// signTypedData. Nothing else is needed for `exact`.const client = new x402Client();registerExactEvmScheme(client, {    signer: { address: account.address, signTypedData: (a: any) => account.signTypedData(a) } as any,});// Unlike batch-settlement, no client.register(NETWORK, ...) call: the exact scheme acts on// whichever network the server's 402 names.const fetchWithPayment = wrapFetchWithPayment(fetch, client);console.log("✅ buyer client ready (exact scheme, stateless)");

## Pre-flight — buyer USDC vs. the advertised price

$0.07 per call (`USDC_PAYMENT_AMOUNT` in `genimg_x402_token.ts`). This checks the buyer only; the
**server wallet's ETH balance** is the other prerequisite and is enforced server-side by
`preFlightChecks` — if it is short you get a 500 naming the deficit, not a payment error.

In [ ]:
import { formatUnits } from "npm:viem@2";const erc20BalanceOfAbi = [{    inputs: [{ name: "account", type: "address" }],    name: "balanceOf",    outputs: [{ name: "", type: "uint256" }],    stateMutability: "view",    type: "function",}] as const;const PRICE_ATOMIC = 70_000n; // $0.07, 6 decimals — matches USDC_PAYMENT_AMOUNT server-side.const buyerUsdc = await publicClient.readContract({    address: config.usdcAddress,    abi: erc20BalanceOfAbi,    functionName: "balanceOf",    args: [account.address],});console.log(`💵 Buyer USDC: ${formatUnits(buyerUsdc, 6)}  (price per image: ${formatUnits(PRICE_ATOMIC, 6)})`);if (buyerUsdc < PRICE_ATOMIC) {    console.log(`⚠️  Not enough USDC. Faucet: ${config.faucet}`);} else {    console.log(`   Covers ${buyerUsdc / PRICE_ATOMIC} more image(s).`);}

## Generate — the paid request

The bare request gets a `402`, the SDK signs an EIP-3009 authorization, the retry carries the
payment header, and the server generates, mints, and settles.

The response is the `images/v1` envelope. **`data[0].url` is the image** — a client that only
wants an image reads that and never touches `x_nft`.

In [ ]:
async function generateImage(body: Record<string, unknown>) {    const started = performance.now();    const response = await fetchWithPayment(SERVICE_URL, {        method: "POST",        headers: { "Content-Type": "application/json" },        body: JSON.stringify(body),    });    const elapsedMs = Math.round(performance.now() - started);    const text = await response.text();    let parsed: any;    try { parsed = JSON.parse(text); } catch { parsed = text; }    console.log(`📡 status=${response.status}  elapsed=${elapsedMs}ms`);    let receipt: unknown = null;    try {        receipt = new x402HTTPClient(client).getPaymentSettleResponse((n: string) => response.headers.get(n));        console.log("🧾 settlement receipt:", JSON.stringify(receipt, null, 2));    } catch (err) {        console.log("🧾 no settlement receipt:", (err as Error).message);    }    return { response, body: parsed, receipt, elapsedMs };}/** * Asserts the images/v1 contract, not just "it worked". A notebook `throw` fails the cell loudly. * Deliberately checks data[0].url separately from x_nft: the first is the contract every * implementer must satisfy, the second is this agent's declared nft-mint capability. */function assertEnvelope(label: string, r: { response: Response; body: any }) {    if (r.response.status !== 200) {        throw new Error(`${label}: expected HTTP 200, got ${r.response.status} — ${JSON.stringify(r.body)}`);    }    const url = r.body?.data?.[0]?.url;    if (typeof url !== "string" || !url.startsWith("http")) {        throw new Error(`${label}: no data[0].url in the envelope — got ${JSON.stringify(r.body).slice(0, 300)}`);    }    if (typeof r.body?.created !== "number") throw new Error(`${label}: missing 'created'`);    if (typeof r.body?.model !== "string") throw new Error(`${label}: missing 'model'`);    const nft = r.body?.x_nft;    if (!nft?.status) throw new Error(`${label}: missing x_nft.status`);    if (nft.status === "mint_failed") {        // Not a failure of this cell: the image is real and unpaid-for. See the next section.        console.log(`⚠️  ${label}: image OK but mint failed — ${nft.reason}`);        console.log("    Nothing was settled; check the server wallet's ETH balance.");        return;    }    if (nft.status !== "minted") throw new Error(`${label}: unexpected x_nft.status ${nft.status}`);    if (typeof nft.token_id !== "number") throw new Error(`${label}: minted but no token_id`);    console.log(`✅ ${label}: model=${r.body.model} token_id=${nft.token_id}`);    console.log(`   image: ${url}`);    console.log(`   token: ${config.explorer}/token/${nft.contract}?a=${nft.token_id}`);}const gen = await generateImage({ prompt: "A lighthouse in a storm, dramatic lighting", size: "1024x1024" });assertEnvelope("generate", gen);

## Render

In [ ]:
const imageUrl = gen.body?.data?.[0]?.url;if (imageUrl) {    const bytes = new Uint8Array(await (await fetch(imageUrl)).arrayBuffer());    // On testnet this is the server's placeholder, not a generated image — that is correct.    Deno.jupyter.display({ "image/jpeg": btoa(String.fromCharCode(...bytes)) }, { raw: true });}

## The mint-failure contract — read the frontend work against this

**A 200 does not mean the NFT was minted.** If generation succeeds but the mint fails, the server
returns 200 with a usable `data[0].url` and:

```json
"x_nft": { "status": "mint_failed", "reason": "..." }
```

Two things follow, and both matter for `ImageGenerator.tsx`:

- **There is no `token_id`.** Today's frontend does `BigInt(result.tokenId)` unconditionally — on
  a `mint_failed` response that is `BigInt(undefined)`, which throws. The guard has to exist
  before the frontend switches to the envelope.
- **Nothing was settled.** No `Payment-Response` header, no charge. This is the one place the
  endpoint answers 200 with no payment settled, and it preserves the behaviour the service has
  always had, since settlement only ever ran after a successful mint.

The likeliest cause on a local run is the server wallet being out of Optimism Sepolia ETH.

## Edit mode — a vendor extension, not part of `images/v1`

`mode` and `referenceImage` are this agent's own fields. They are deliberately outside the
interop floor: image *editing* is not part of the OpenAI images-generation body, and requiring it
would exclude generation-only providers.

⚠️ Each call is another $0.07 and another mint.

In [ ]:
// Uses the image generated above as the reference, so the cell is self-contained.const srcUrl = gen.body?.data?.[0]?.url;if (!srcUrl) throw new Error("no image from the previous cell to edit");const srcBytes = new Uint8Array(await (await fetch(srcUrl)).arrayBuffer());const referenceImage = btoa(String.fromCharCode(...srcBytes));const edited = await generateImage({    prompt: "Make it sunrise, warm golden light",    size: "1024x1024",    mode: "edit",    referenceImage,});assertEnvelope("edit", edited);

## Optional — the strict request contract

⚠️ **This cell costs $0.07.** The 402 challenge deliberately comes *before* request validation, so
a client can always discover the payment terms it is supposed to satisfy — which means an unpaid
junk request returns a 402, not a 400. Seeing the validation error requires paying for it.

Unknown fields are rejected rather than ignored: this endpoint charges per call, so silently
dropping a `quality: "hd"` the caller believed in would mean taking money for a request we did not
fulfil as asked.

In [ ]:
const rejected = await generateImage({ prompt: "a cat", quality: "hd" });console.log(JSON.stringify(rejected.body, null, 2));if (rejected.response.status !== 400) throw new Error(`expected 400, got ${rejected.response.status}`);if (rejected.body?.error?.param !== "quality") {    throw new Error(`expected error.param 'quality', got ${JSON.stringify(rejected.body?.error)}`);}console.log("✅ unknown field rejected, and the error names it");